In [ ]:
# Databricks notebook source
import sys
sys.path.append("../data_quality")
from pyspark.sql import functions as F
from dq_utils import ensure_log_table_exists, log_check, check_value_in_range

spark.sql("USE CATALOG brazilian_ecommerce")
ensure_log_table_exists(spark)

gold_sales_summary = spark.table("prod.gold_sales_summary")
gold_revenue_by_category_month = spark.table("prod.gold_revenue_by_category_month")
gold_revenue_by_seller = spark.table("prod.gold_revenue_by_seller")
gold_delivery_performance = spark.table("prod.gold_delivery_performance")
gold_delivery_by_region = spark.table("prod.gold_delivery_by_region")
gold_customer_summary = spark.table("prod.gold_customer_summary")
gold_product_performance = spark.table("prod.gold_product_performance")
gold_category_review_scores = spark.table("prod.gold_category_review_scores")

# --- domain/range checks ---

check_value_in_range(spark, "gold", "gold_sales_summary", gold_sales_summary,
                       "total_order_value", 0, 1_000_000, raise_on_failure=True)

check_value_in_range(spark, "gold", "gold_revenue_by_category_month", gold_revenue_by_category_month,
                       "total_revenue", 0, 1_000_000_000, raise_on_failure=True)

check_value_in_range(spark, "gold", "gold_revenue_by_seller", gold_revenue_by_seller,
                       "pct_delayed_orders", 0, 1, raise_on_failure=True)
check_value_in_range(spark, "gold", "gold_revenue_by_seller", gold_revenue_by_seller,
                       "pct_shipped_late_single_seller", 0, 1, raise_on_failure=True)

check_value_in_range(spark, "gold", "gold_delivery_performance", gold_delivery_performance,
                       "pct_on_time", 0, 1, raise_on_failure=True)
check_value_in_range(spark, "gold", "gold_delivery_performance", gold_delivery_performance,
                       "pct_delayed", 0, 1, raise_on_failure=True)

check_value_in_range(spark, "gold", "gold_delivery_by_region", gold_delivery_by_region,
                       "pct_delayed", 0, 1, raise_on_failure=True)

check_value_in_range(spark, "gold", "gold_customer_summary", gold_customer_summary,
                       "avg_review_score_given", 1, 5, raise_on_failure=True)

check_value_in_range(spark, "gold", "gold_product_performance", gold_product_performance,
                       "avg_review_score", 1, 5, raise_on_failure=True)

check_value_in_range(spark, "gold", "gold_category_review_scores", gold_category_review_scores,
                       "avg_review_score", 1, 5, raise_on_failure=True)
check_value_in_range(spark, "gold", "gold_category_review_scores", gold_category_review_scores,
                       "pct_low_reviews", 0, 1, raise_on_failure=True)

# # --- reconciliation: total revenue should roughly match across two independently-built gold tables ---

# sales_summary_total = gold_sales_summary.agg(F.sum("total_order_value")).first()[0]
# by_category_total = gold_revenue_by_category_month.agg(F.sum("total_revenue")).first()[0]
# diff_pct = abs(sales_summary_total - by_category_total) / sales_summary_total

# log_check(spark, "gold", "cross_table_reconciliation", "revenue_matches_sales_summary_vs_category",
#            diff_pct < 0.01,
#            details=f"sales_summary={sales_summary_total}, by_category={by_category_total}, diff={diff_pct:.4f}",
#            raise_on_failure=True)

# # --- reconciliation: order count should roughly match across gold_sales_summary vs gold_customer_summary ---

# sales_summary_orders = gold_sales_summary.select("order_id").distinct().count()
# customer_summary_orders = gold_customer_summary.agg(F.sum("total_orders")).first()[0]
# order_diff_pct = abs(sales_summary_orders - customer_summary_orders) / sales_summary_orders

# log_check(spark, "gold", "cross_table_reconciliation", "order_count_matches_sales_summary_vs_customer_summary",
#            order_diff_pct < 0.01,
#            details=f"sales_summary={sales_summary_orders}, customer_summary={customer_summary_orders}, diff={order_diff_pct:.4f}",
#            raise_on_failure=True)